
# 練習問題: 本当にGPUで実行されているか確かめる

## 目標

`#pragma omp target` (Fortran では `!$omp target` ... `!$omp end target`) は, GPUが使えないときには **黙ってCPUにフォールバック** して実行される.
そのため「GPUで動かしているつもりが, 実はCPUで動いていた」という事故が起こりうる.

そこで `omp_is_initial_device()` を使って, ある領域が実際にどこで実行されたかを判定する方法と, 環境変数 `OMP_TARGET_OFFLOAD` で実行場所を制御する方法を体験する.

- `omp_is_initial_device()` は, **ホスト(CPU)上では 1**, **デバイス(GPU)上では 0** を返す.

## 課題

`where_am_i.cpp` (または `where_am_i.f90`) は, まずホスト上で `omp_is_initial_device()` の値を表示し, 続いて `target` 領域の中で同じ関数の値を表示する.
コメント `TODO` の指示に従って **OpenMP の指示行を1つ追加** し, 後半の表示が `target` 領域(GPU上)で実行されるようにせよ.

- C++: メッセージを表示するブロック `{ ... }` の直前に `#pragma omp target` を1行加える.
- Fortran: `print` 文を `!$omp target` と `!$omp end target` で囲む.

表示するだけなので `map` 節を考える必要はない. それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -mp=gpu where_am_i.cpp -o where_am_i.exe

# Fortran
nvfortran -mp=gpu where_am_i.f90 -o where_am_i.exe
```

GPUは計算ノードにのみ搭載されているので, `%%bash_submit` でジョブとして投入して実行する.
以下の **3通り** の方法で実行し, `inside target:` の行に表示される値を比べよ.

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

# (1) 既定の動作 (GPUがあればGPU, なければCPU)
./where_am_i.exe

# (2) GPUでの実行を強制 (GPUが使えなければエラー)
OMP_TARGET_OFFLOAD=MANDATORY ./where_am_i.exe

# (3) 必ずCPUで実行
OMP_TARGET_OFFLOAD=DISABLED ./where_am_i.exe
```

## 期待される結果

- `inside target:` の値が **0** なら, その領域は **GPU** で実行されている.
- `inside target:` の値が **1** なら, その領域は **CPU** で実行されている (フォールバック).
- `on host:` の値は常に 1 (こちらは必ずホストで実行されるため).

計算ノードでの想定:

- (2) `MANDATORY` では `inside target:` は **0** になる (GPUが使われている).
- (3) `DISABLED` では `inside target:` は **1** になる (CPUに強制).
- (1) 既定では, 計算ノードにGPUがあれば **0** になるはず.

このように, `printf` / `print` の出力だけでは区別がつかない「どこで実行されたか」を, `omp_is_initial_device()` で確実に判定できる.
GPU向けのコードを書いたら, 思い込みで終わらせず実際にGPUで動いていることを確認する習慣をつけよう.



# 1. AIチューター
- 以下は必要に応じて実行（毎度実行する必要はない）


In [ ]:
import heytutor


## 1-1. 一般的な質問
- ChatGPTなどに聞くときのように自由に質問可能。
- ただし「答えを教えて」などは自制すること。


In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 1-2. この問題に関するヒント
- `{file:problem.md}` は上記の問題文に展開される。


In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}


## 1-3. いくつかの変数
* それぞれ以下のように展開される。

* `{file:FILENAME}` : _FILENAME_ の中身
* `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前の入力・出力, etc.

## 1-4. 困ったときのヘルプ
* コンパイル時や実行時のエラー直後に以下を実行するとエラーに関するヘルプが得られる。


In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:where_am_i.cpp}

コマンドと実行結果:
{bash[-1]}



## 1-5. フィードバック
* 答えが出た後も、無駄なところや、より良いやり方がないかを聞くことを推奨。
* 以下のファイル名は適宜書き換えよ (Fortranなら `.cpp` -> `.f90` とするなど)


In [ ]:
%%hey

フィードバックを下さい。

問題:
{file:problem.md}

私の答:
{file:where_am_i.cpp}


# 2. ジョブ投入ツール
- 以下を実行しておくと、`%%bash_submit_a` (Aquariousへのジョブ投入), `%%bash_submit_o` (Odyssey へのジョブ投入) が使えるようになる


In [ ]:
import wisteria_submit

# 3. C++ ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ where_am_i.cpp
#include <cstdio>
#include <omp.h>

int main() {
  printf("on host: omp_is_initial_device() = %d\n", omp_is_initial_device());
  // TODO: 下のブロックの直前に #pragma omp target を1行追加し, ブロックの中身をデバイス(GPU)上で実行させよ. (表示するだけなので map 節は不要)
  {
    printf("inside target: omp_is_initial_device() = %d\n", omp_is_initial_device());
  }
  return 0;
}

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=gpu where_am_i.cpp -o where_am_i_cpp.exe

## 3-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./where_am_i_cpp.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a

./where_am_i_cpp.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./where_am_i_cpp.exe

## 3-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:where_am_i.cpp}


# 4. Fortran ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ where_am_i.f90
program where_am_i
  use omp_lib
  print "(a,i0)", "on host: omp_is_initial_device() = ", omp_is_initial_device()
  ! TODO: 下の print を !$omp target ... !$omp end target で囲み, デバイス(GPU)上で実行させよ. (表示するだけなので map 節は不要)
  print "(a,i0)", "inside target: omp_is_initial_device() = ", omp_is_initial_device()
  ! TODO: 上で始めた target 領域を閉じる (!$omp end target).
end program where_am_i

## 4-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=gpu where_am_i.f90 -o where_am_i_f90.exe

## 4-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./where_am_i_f90.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a
./where_am_i_f90.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit

#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./where_am_i_f90.exe

## 4-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:where_am_i.f90}